# Global B2B Dataset Clustering – Improved Version

This notebook builds on the original clustering example and adds a few enhancements to obtain more meaningful clusters:
- Cleaner title preprocessing
- Optionally combine sentence‑transformer embeddings with TF‑IDF features
- Standard‑scale the feature matrix
- Dimensionality reduction with PCA before clustering
- Expanded K‑Means search (more clusters, more restarts)
- Additional clustering alternatives (Agglomerative, HDBSCAN)
- Extra evaluation metrics (Calinski‑Harabasz, Davies‑Bouldin)

The heavy lifting (sentence embeddings) still uses a small hosted model, so no large local model is required.

In [ ]:
%pip install -q pandas numpy scikit-learn umap-learn sentence-transformers hdbscan joblib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from sentence_transformers import SentenceTransformer
import re
import warnings
warnings.filterwarnings('ignore')
sns.set(style='whitegrid')

## Load the CSV dataset
Upload `globalb2bdataset_cleaned.csv` to the Colab session (or mount Google Drive) and adjust the path below.

In [ ]:
csv_path = '/content/globalb2bdataset_cleaned.csv'  # change if using Drive
df = pd.read_csv(csv_path)
df.head()

## Basic cleaning – keep rows with required columns

In [ ]:
df_clean = df.dropna(subset=['decision_maker_title', 'industry']).reset_index(drop=True)
df_clean.shape

## Title preprocessing function

In [ ]:
def clean_title(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # drop punctuation
    text = re.sub(r'\s+', ' ', text).strip()   # collapse spaces
    return text

titles_raw = df_clean['decision_maker_title'].tolist()
titles = [clean_title(t) for t in titles_raw]

## Sentence embeddings (lightweight model)

In [ ]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'  # ~20 MB, good trade‑off
embedder = SentenceTransformer(model_name)
title_emb = embedder.encode(titles, batch_size=64, show_progress_bar=True)
title_emb.shape

## Optional: TF‑IDF features for the titles (captures rare tokens)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=500, stop_words='english')
title_tfidf = tfidf.fit_transform(titles).toarray()
title_tfidf.shape

## Encode the categorical *industry* column (one‑hot)

In [ ]:
industry_enc = OneHotEncoder(sparse=False, handle_unknown='ignore')
industry_onehot = industry_enc.fit_transform(df_clean[['industry']])
industry_onehot.shape

## Assemble the feature matrix
We concatenate: sentence embeddings + TF‑IDF + industry one‑hot.

In [ ]:
X_raw = np.hstack([title_emb, title_tfidf, industry_onehot])
print('Raw feature shape:', X_raw.shape)

## Standard‑scale the features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
print('Scaled shape:', X_scaled.shape)

## Dimensionality reduction with PCA (optional but often helpful)

In [ ]:
pca = PCA(n_components=80, random_state=42)  # keep ~80 components (adjust as needed)
X_pca = pca.fit_transform(X_scaled)
print('PCA shape:', X_pca.shape)

## Visualise with UMAP (2‑D)

In [ ]:
reducer = umap.UMAP(n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_pca)
plt.figure(figsize=(8,6))
sns.scatterplot(x=X_umap[:,0], y=X_umap[:,1], hue=df_clean['industry'], palette='tab10', s=60)
plt.title('UMAP of PCA‑reduced features (colored by industry)')
plt.show()

## Clustering – expanded K‑Means search

In [ ]:
def eval_metrics(labels, data):
    if len(set(labels)) <= 1:
        return -1, -1, -1
    sil = silhouette_score(data, labels)
    ch = calinski_harabasz_score(data, labels)
    db = davies_bouldin_score(data, labels)
    return sil, ch, db

best_k, best_score = None, -1
best_model = None
for k in range(2, 31):
    km = KMeans(n_clusters=k, random_state=42, n_init=20, init='k-means++')
    km_labels = km.fit_predict(X_pca)
    sil, ch, db = eval_metrics(km_labels, X_pca)
    if sil > best_score:
        best_score, best_k = sil, k
        best_model = km
        best_labels = km_labels
print(f'Best K‑Means: k={best_k}, silhouette={best_score:.3f}')

## Alternative clustering algorithms (optional)

In [ ]:
# Agglomerative (Ward) – use same k range
best_agg_score = -1
for k in range(2, 31):
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    agg_labels = agg.fit_predict(X_pca)
    sil, _, _ = eval_metrics(agg_labels, X_pca)
    if sil > best_agg_score:
        best_agg_score = sil
        best_agg_k = k
        best_agg = agg
        best_agg_labels = agg_labels
print(f'Best Agglomerative: k={best_agg_k}, silhouette={best_agg_score:.3f}')

# HDBSCAN – automatically determines cluster count
import hdbscan
hdb = hdbscan.HDBSCAN(min_cluster_size=10, metric='euclidean')
hdb_labels = hdb.fit_predict(X_pca)
if len(set(hdb_labels)) > 1:
    sil, _, _ = eval_metrics(hdb_labels, X_pca)
    print('HDBSCAN silhouette:', sil)
else:
    print('HDBSCAN produced only noise')

## Choose the best algorithm based on silhouette (you can also consider CH or DB scores)

In [ ]:
candidates = {
    'kmeans': (best_score, best_labels, best_model),
    'agg': (best_agg_score, best_agg_labels, best_agg)
}
if 'hdb_labels' in locals() and len(set(hdb_labels)) > 1:
    sil_hdb, _, _ = eval_metrics(hdb_labels, X_pca)
    candidates['hdbscan'] = (sil_hdb, hdb_labels, hdb)

best_algo = max(candidates, key=lambda k: candidates[k][0])
best_sil, final_labels, final_model = candidates[best_algo]
print('Selected algorithm:', best_algo, 'silhouette =', best_sil)

## Visualise the final clusters on the UMAP projection

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x=X_umap[:,0], y=X_umap[:,1], hue=final_labels, palette='viridis', s=60, legend=False)
plt.title(f'Clusters by {best_algo.upper()} (silhouette={best_sil:.3f})')
plt.show()

## Save artefacts for deployment
The scaler, PCA, industry encoder and the chosen clustering model are persisted so the same pipeline can be re‑used on new data.

In [ ]:
import os
out_dir = 'clustering_model_v3'
os.makedirs(out_dir, exist_ok=True)
joblib.dump(scaler, os.path.join(out_dir, 'scaler.joblib'))
joblib.dump(pca, os.path.join(out_dir, 'pca.joblib'))
joblib.dump(industry_enc, os.path.join(out_dir, 'industry_encoder.joblib'))
joblib.dump(final_model, os.path.join(out_dir, 'clusterer.joblib'))
with open(os.path.join(out_dir, 'embedder_name.txt'), 'w') as f:
    f.write(model_name)
print('Saved deployment artefacts to', out_dir)

## Deployment suggestions (same as before)
- Vertex AI (batch / online)
- Cloud Run container
- Hugging Face Spaces (Gradio)
- FastAPI + Cloud Functions
All of them can load the artefacts saved above and call the HuggingFace inference API for embeddings, keeping the serving cost low.